In [ ]:
## Tradtional Streaming DStream 
## Spark Context 
## Stream Context 
from pyspark import SparkContext 
from pyspark.streaming import StreamingContext

In [ ]:
## Create both SparkContext and Stream Context

sc = SparkContext("local[2]", "StreamingAppDemo")
ssc = StreamingContext(sc, 1)

# 1 sec sampling rate -> create a micro batch with 1 second of input data

In [ ]:
# Create a socket stream input 
lines = ssc.socketTextStream("localhost", 9999) # ip address, port 

In [ ]:
"""
input: how was your day my day was good good is better

output: 
    how - 1
    was - 2
    your - 1
    day - 2
    my - 1
    good - 2
    is - 1
    better - 1
"""
""" 
spark batch -> 
1. split the string based on the delimiter space
    [how, was, your, day, my, day, was, good, good, is, better]

2. map with repetitions 
    [(how, 1), (was, 1), (your, 1), (day, 1), ... (better, 1)]

3. Group by the word and sum up all weightage 
    [(how, 1), (was, 2), ... (better, 1)]
"""

wordsRdd = lines.flatMap(lambda l: l.split(" ")) # creates a 1D array from single string

wordsWithWeightRdd = wordsRdd.map(lambda x: (x, 1)) # key, value -> key -> [1, 1, 1, ...]

wordCount = wordsWithWeightRdd.reduceByKey(lambda x, y: x + y)

# give wordCount as output
wordCount.pprint()

In [ ]:
# This is a streaming job, we need to listen to the stream and stop when the stream stops 
ssc.start()
ssc.awaitTermination()